## Cleaning Puma

Upload
1. puma_data_2023.csv
2. us-state-ansi-fips.csv
3. ipums_puma_2020_ipums_puma_2020.shp.csv
4. the file named 'ipums_puma_2020'

Output
1. the filed named 'puma_data_2023_integrated_spatial'

In [14]:
import pandas as pd
import numpy as np

# Load main PUMA data
puma_data = pd.read_csv('puma_data_2023.csv')

# Load state mapping file
state_map = pd.read_csv('us-state-ansi-fips.csv')

# Load PUMA geographic mapping file
puma_geo = pd.read_csv('ipums_puma_2020_ipums_puma_2020.shp.csv')

print(f"Loaded {len(puma_data)} PUMA records")
print(f"Loaded {len(state_map)} state records")
print(f"Loaded {len(puma_geo)} PUMA geographic records")

Loaded 2462 PUMA records
Loaded 51 state records
Loaded 2486 PUMA geographic records


In [15]:
# Convert GEOID to string and zero-pad to 7 digits to handle both 6 and 7 digit GEOIDs
# This ensures GEOIDs like 100100 become "0100100" and we can correctly extract state FIPS
puma_data['GEOID_str'] = puma_data['GEOID'].astype(str).str.zfill(7)

# Extract first 2 digits as state FIPS code from the zero-padded GEOID
puma_data['State_FIPS'] = puma_data['GEOID_str'].str[:2]

In [16]:
# Merge with state mapping to get state names

# Ensure column names are stripped of whitespace (safety check)
state_map.columns = state_map.columns.str.strip()

# Convert state FIPS to string for merging
state_map['st'] = state_map['st'].astype(str).str.zfill(2)

# Merge state information
puma_data = puma_data.merge(
    state_map[['st', 'stname', 'stusps']],
    left_on='State_FIPS',
    right_on='st',
    how='left'
)

# Rename columns for clarity
puma_data.rename(columns={
    'stname': 'State_Name',
    'stusps': 'State_Abbreviation'
}, inplace=True)

In [17]:
# Merging with PUMA geographic mapping

# Convert GEOID to string and ensure matching format (zero-pad to 7 digits)
# Use the already zero-padded GEOID_str if it exists, otherwise pad it
if 'GEOID_str' not in puma_data.columns:
    puma_data['GEOID_str'] = puma_data['GEOID'].astype(str).str.zfill(7)
puma_data['GEOID'] = puma_data['GEOID_str']  # Use the zero-padded version
puma_geo['Geoid'] = puma_geo['Geoid'].astype(str).str.zfill(7)

# Merge PUMA geographic information
puma_data = puma_data.merge(
    puma_geo[['Geoid', 'Name', 'State', 'Puma']],
    left_on='GEOID',
    right_on='Geoid',
    how='left'
)

# Rename columns for clarity
puma_data.rename(columns={
    'Name': 'PUMA_Name',
    'State': 'PUMA_State_Name',
    'Puma': 'PUMA_Code'
}, inplace=True)

# Remove trailing "PUMA" text from names
puma_data['PUMA_Name'] = (
    puma_data['PUMA_Name']
    .fillna('')
    .str.replace(r'\s*PUMA$', '', regex=True)
    .str.strip()
)

# Drop duplicate columns and temporary columns
puma_data.drop(['st', 'Geoid', 'GEOID_str'], axis=1, inplace=True, errors='ignore')



In [18]:
# Create filter flags for non-mutually exclusive filtering

# Define demographic categories and their corresponding percentage columns
demographic_categories = {
    'White': 'White_percent',
    'Black': 'Black_percent',
    'AIAN': 'AIAN_percent',  # American Indian/Alaska Native
    'Asian': 'Asian_percent',
    'NHOPI': 'NHOPI_percent',  # Native Hawaiian/Other Pacific Islander
    'Hispanic': 'Hispanic_percent',
    'Other': 'Other_percent',
    'Multi': 'Multi_percent'
}

# Create binary flags for each demographic category (1 if data exists and > 0)
for category, percent_col in demographic_categories.items():
    if percent_col in puma_data.columns:
        # Create flag: 1 if percentage > 0, 0 otherwise
        puma_data[f'{category}_Flag'] = (puma_data[percent_col] > 0).astype(int)
        # Also create a categorical version for easier filtering
        puma_data[f'{category}_Category'] = puma_data[percent_col].apply(
            lambda x: category if pd.notna(x) and x > 0 else 'Not ' + category
        )

# Create flags for economic categories
if 'Poordu_percent' in puma_data.columns:
    puma_data['Poor_Flag'] = (puma_data['Poordu_percent'] > 0).astype(int)
    puma_data['Poor_Category'] = puma_data['Poordu_percent'].apply(
        lambda x: 'Poor' if pd.notna(x) and x > 0 else 'Not Poor'
    )


In [19]:
# Create combined demographic flags for intersection analysis

# Create flags for ALL demographic groups × Poor combinations
for category in demographic_categories.keys():
    percent_col = demographic_categories[category]
    if percent_col in puma_data.columns and 'Poordu_percent' in puma_data.columns:
        flag_name = f'{category}_Poor_Flag'
        puma_data[flag_name] = (
            (puma_data[percent_col] > 0) &
            (puma_data['Poordu_percent'] > 0)
        ).astype(int)
        print(f"  Created {flag_name}")

# Note: Renter flag combinations are not created as they will be dropped later

print(f"\nTotal combined flags created: {len([c for c in puma_data.columns if '_Flag' in c and '_Poor' in c])}")

  Created White_Poor_Flag
  Created Black_Poor_Flag
  Created AIAN_Poor_Flag
  Created Asian_Poor_Flag
  Created NHOPI_Poor_Flag
  Created Hispanic_Poor_Flag
  Created Other_Poor_Flag
  Created Multi_Poor_Flag

Total combined flags created: 8


In [20]:
# Clean and standardize column names

# Replace NaN values with 0 for percentage columns (where appropriate)
percent_cols = [col for col in puma_data.columns if '_percent' in col]
for col in percent_cols:
    puma_data[col] = puma_data[col].fillna(0)

# Replace NaN values with 0 for frequency columns
freq_cols = [col for col in puma_data.columns if 'Frequency' in col]
for col in freq_cols:
    puma_data[col] = puma_data[col].fillna(0)


In [21]:
# Reorder columns for better Tableau usability

# Define column order (geographic info first, then metrics, then flags)
geographic_cols = ['GEOID', 'State_FIPS', 'State_Name', 'State_Abbreviation',
                   'PUMA_Code', 'PUMA_Name', 'PUMA_State_Name']

# Get all other columns
other_cols = [col for col in puma_data.columns if col not in geographic_cols]

# Reorder: geographic first, then others
final_cols = geographic_cols + other_cols

# Only include columns that exist
final_cols = [col for col in final_cols if col in puma_data.columns]

puma_data = puma_data[final_cols]


In [22]:
# Drop columns according to specified rules

print("Applying column dropping rules...")
all_columns = puma_data.columns.tolist()
columns_to_drop = []

# Rule 1: Delete PUMA_State_Name
if 'PUMA_State_Name' in all_columns:
    columns_to_drop.append('PUMA_State_Name')

# Rule 2: Delete {Race}_renter_flag (e.g., White_Renter_Flag, Black_Renter_Flag, etc.)
race_renter_flags = [col for col in all_columns if col.endswith('_Renter_Flag')]
columns_to_drop.extend(race_renter_flags)

# Rule 3: Delete specific error columns (StdErr only, keep MOE columns)
error_columns = [
    'StdErrWgtFreq', 'StdErrofRowPercent',
    'Poordu_StdErrWgtFreq', 'Poordu_StdErrofRowPercent'
]
for col in error_columns:
    if col in all_columns:
        columns_to_drop.append(col)

# Rule 4: Delete any columns starting with 'Renter_'
renter_columns = [col for col in all_columns if col.startswith('Renter_')]
columns_to_drop.extend(renter_columns)

# Rule 5: Among DU by race, only keep {Race}_DU_WeightedFrequency, {Race}_percent, {Race}_CV_Percent
race_prefixes = ['White', 'Black', 'AIAN', 'Asian', 'NHOPI', 'Hispanic', 'Other', 'Multi', 'Hisp']

for race in race_prefixes:
    race_columns = [col for col in all_columns if col.startswith(f'{race}_')]

    keep_du_cols = [
        f'{race}_DU_WeightedFrequency',
        f'{race}_percent',
        f'{race}_CV_Percent'
    ]

    for col in race_columns:
        if col in columns_to_drop or '_Flag' in col or '_Category' in col or 'Poordu' in col:
            continue

        if col in keep_du_cols:
            continue

        if '_DU_' in col:
            columns_to_drop.append(col)
        elif any(x in col for x in ['StdErrWgtFreq', 'StdErrofRowPercent']):
            if col not in columns_to_drop:
                columns_to_drop.append(col)

# Rule 6: Among Poordu by Race, only keep {Race}_Poordu_WeightedFrequency, {Race}_Poordu_percent, {Race}_Poordu_CV_Percent, {Race}_Poordu_MOE_freq, {Race}_Poordu_MOE_percent
for race in race_prefixes:
    poordu_columns = [col for col in all_columns if col.startswith(f'{race}_') and 'Poordu' in col]

    keep_poordu_cols = [
        f'{race}_Poordu_WeightedFrequency',
        f'{race}_Poordu_percent',
        f'{race}_Poordu_CV_Percent',
        f'{race}_Poordu_MOE_freq',
        f'{race}_Poordu_MOE_percent'
    ]

    for poordu_col in poordu_columns:
        if poordu_col not in keep_poordu_cols:
            columns_to_drop.append(poordu_col)

# Rule 7: Keep flags, but no Renter_flag
renter_flags = [col for col in all_columns if 'Flag' in col and 'Renter' in col]
columns_to_drop.extend(renter_flags)

# Remove duplicates and drop columns
columns_to_drop = list(set(columns_to_drop))
columns_to_drop = [col for col in columns_to_drop if col in all_columns]

print(f"Dropping {len(columns_to_drop)} columns...")
puma_data = puma_data.drop(columns=columns_to_drop, errors='ignore')

print(f"Final shape: {puma_data.shape}")
print(f"Final columns: {len(puma_data.columns)}")


Applying column dropping rules...
Dropping 44 columns...
Final shape: (2462, 122)
Final columns: 122


In [23]:
# Rename columns according to specified rules
print("\nRenaming columns...")
column_mapping = {}

# 1) Direct mappings
column_mapping['State_Abbreviation'] = 'State_Abbr'
column_mapping['DU_WeightedFrequency'] = 'DU_Fre'
column_mapping['DUpercent_prevalenceinPUMA'] = 'DU_Per'
column_mapping['CV_Percent'] = 'DU_CV'
column_mapping['Poordu_WeightedFrequency'] = 'PoDU_Fre'
column_mapping['Poordu_percent'] = 'PoDU_Per'
column_mapping['Poordu_CV_Percent'] = 'PoDU_CV'
column_mapping['Poor_Category'] = 'Poor_Cate'

# Define all race names found in the data
races = ['White', 'Black', 'AIAN', 'Asian', 'NHOPI', 'Hispanic', 'Other', 'Multi']

# 2) Race columns - use first two characters
# Pattern: {Race}_DU_WeightedFrequency -> {Race_abbr}_Fre
# Pattern: {Race}_percent -> {Race_abbr}_per
# Pattern: {Race}_CV_Percent -> {Race_abbr}_CV
for race in races:
    race_abbr = race[:2]  # First two characters
    column_mapping[f'{race}_DU_WeightedFrequency'] = f'{race_abbr}_Fre'
    column_mapping[f'{race}_percent'] = f'{race_abbr}_per'
    column_mapping[f'{race}_CV_Percent'] = f'{race_abbr}_CV'

# 3) Poor by Race columns - use first two characters
# Pattern: {Race}_Poordu_WeightedFrequency -> {Race_abbr}_Po_Fre
# Pattern: {Race}_Poordu_percent -> {Race_abbr}_Po_per
# Pattern: {Race}_Poordu_CV_Percent -> {Race_abbr}_Po_CV
for race in races:
    race_abbr = race[:2]  # First two characters
    column_mapping[f'{race}_Poordu_WeightedFrequency'] = f'{race_abbr}_Po_Fre'
    column_mapping[f'{race}_Poordu_percent'] = f'{race_abbr}_Po_per'
    column_mapping[f'{race}_Poordu_CV_Percent'] = f'{race_abbr}_Po_CV'

# Handle "Hisp" variant (found in the data)
column_mapping['Hisp_Poordu_WeightedFrequency'] = 'Hi_Po_Fre'
column_mapping['Hisp_Poordu_percent'] = 'Hi_Po_per'
column_mapping['Hisp_Poordu_CV_Percent'] = 'Hi_Po_CV'

# 4) Category columns - use first two characters
# Pattern: {Race}_Category -> {Race_abbr}_Cate
for race in races:
    race_abbr = race[:2]  # First two characters
    column_mapping[f'{race}_Category'] = f'{race_abbr}_Cate'

# 5) Flag columns - use first two characters
# Pattern: {Race}_Flag -> {Race_abbr}_Flag
for race in races:
    race_abbr = race[:2]  # First two characters
    column_mapping[f'{race}_Flag'] = f'{race_abbr}_Flag'

# 6) Poor_Flag columns - use first two characters
# Pattern: {Race}_Poor_Flag -> {Race_abbr}_Po_Flag
for race in races:
    race_abbr = race[:2]  # First two characters
    column_mapping[f'{race}_Poor_Flag'] = f'{race_abbr}_Po_Flag'

# 7) MOE column renaming
# Direct MOE mappings
column_mapping['MOE_percent'] = 'MOE_perc'
column_mapping['Poordu_MOE_freq'] = 'Po_MOE_fre'
column_mapping['Poordu_MOE_percent'] = 'Po_MOE_per'

# Race MOE columns - use first two characters
# Pattern: {Race}_MOE_freq -> {Race_abbr}_MOE_fre
# Pattern: {Race}_MOE_percent -> {Race_abbr}_MOE_per
for race in races:
    race_abbr = race[:2]  # First two characters
    column_mapping[f'{race}_MOE_freq'] = f'{race_abbr}_MOE_fre'
    column_mapping[f'{race}_MOE_percent'] = f'{race_abbr}_MOE_per'

# Race Poordu MOE columns - use first two characters
# Pattern: {Race}_Poordu_MOE_freq -> {Race_abbr}_PoM_fre
# Pattern: {Race}_Poordu_MOE_percent -> {Race_abbr}_PoM_per
for race in races:
    race_abbr = race[:2]  # First two characters
    column_mapping[f'{race}_Poordu_MOE_freq'] = f'{race_abbr}_PoM_fre'
    column_mapping[f'{race}_Poordu_MOE_percent'] = f'{race_abbr}_PoM_per'

# Handle "Hisp" variant for MOE columns
column_mapping['Hisp_MOE_freq'] = 'Hi_MOE_fre'
column_mapping['Hisp_MOE_percent'] = 'Hi_MOE_per'
column_mapping['Hisp_Poordu_MOE_freq'] = 'Hi_PoM_fre'
column_mapping['Hisp_Poordu_MOE_percent'] = 'Hi_PoM_per'

# Apply renaming (only rename columns that exist)
existing_mapping = {k: v for k, v in column_mapping.items() if k in puma_data.columns}
puma_data.rename(columns=existing_mapping, inplace=True)
print(f"  Renamed {len(existing_mapping)} columns")



Renaming columns...
  Renamed 115 columns


In [24]:
# Data validation and summary

print(f"\nTotal records: {len(puma_data)}")
print(f"Total columns: {len(puma_data.columns)}")

print("\nGeographic coverage:")
print(f"  States: {puma_data['State_Name'].nunique()}")
print(f"  PUMA areas: {puma_data['GEOID'].nunique()}")

print("\nMissing values:")
missing = puma_data.isnull().sum()
missing = missing[missing > 0]
if len(missing) > 0:
    print(missing.head(10))
else:
    print("  No missing values in key columns")

print("\nDemographic categories with data:")
for category in demographic_categories.keys():
    flag_col = f'{category}_Flag'
    if flag_col in puma_data.columns:
        count = puma_data[flag_col].sum()
        print(f"  {category}: {count} PUMA areas ({count/len(puma_data)*100:.1f}%)")

print("\nCombined flags summary:")
# Count flags by type
poor_flags = [c for c in puma_data.columns if '_Poor_Flag' in c]
print(f"  Demographic × Poor flags: {len(poor_flags)}")

print("\nSample of cleaned data:")
print(puma_data.head())


Total records: 2462
Total columns: 122

Geographic coverage:
  States: 51
  PUMA areas: 2462

Missing values:
PUMA_Code       1
MOE_freq        1
DU_Per          1
DU_CV           1
Po_MOE_fre      2
PoDU_CV         2
Wh_MOE_fre      7
Wh_CV           7
Bl_MOE_fre    625
Bl_CV         625
dtype: int64

Demographic categories with data:

Combined flags summary:
  Demographic × Poor flags: 0

Sample of cleaned data:
     GEOID State_FIPS State_Name State_Abbr  PUMA_Code  \
0  0100100         01    Alabama         AL      100.0   
1  0100200         01    Alabama         AL      200.0   
2  0100300         01    Alabama         AL      300.0   
3  0100400         01    Alabama         AL        NaN   
4  0100402         01    Alabama         AL      402.0   

                                           PUMA_Name  DU_Fre    MOE_freq  \
0            Lauderdale, Colbert & Franklin Counties  1995.0  808.939130   
1                                   Limestone County   167.0  205.725477   
2   

In [25]:
output_file = 'puma_data_2023_cleaned_tableau.csv'

print(f"\nExporting cleaned data to {output_file}...")
puma_data.to_csv(output_file, index=False)


Exporting cleaned data to puma_data_2023_cleaned_tableau.csv...


In [26]:
# Integrate PUMA data with spatial shapefile for Tableau

try:
    import geopandas as gpd
    from pathlib import Path
    import os

    # File paths
    shapefile_path = "ipums_puma_2020.shp"
    output_dir = "puma_data_2023_integrated_spatial"
    output_shp_name = "puma_data_2023_integrated_spatial.shp"

    # Create output directory if it doesn't exist
    os.makedirs(output_dir, exist_ok=True)
    output_path = os.path.join(output_dir, output_shp_name)

    print("\n1. Reading shapefile...")
    # Read the shapefile
    gdf = gpd.read_file(shapefile_path)

    print(f"   ✓ Shapefile loaded: {len(gdf)} records")
    print(f"   ✓ Shapefile columns: {list(gdf.columns)}")
    print(f"   ✓ Coordinate Reference System: {gdf.crs}")

    # Ensure GEOID is string type
    gdf['GEOID'] = gdf['GEOID'].astype(str)

    puma_df = puma_data

    # Ensure GEOID is string type for proper joining
    # CSV GEOID should already be in 7-digit format, but ensure it's string
    puma_df['GEOID'] = puma_df['GEOID'].astype(str).str.zfill(7)

    print("Checking for unmatched records...")
    # Find unmatched records before merging
    shapefile_geoids = set(gdf['GEOID'])
    csv_geoids = set(puma_df['GEOID'])

    in_shapefile_not_csv = shapefile_geoids - csv_geoids
    in_csv_not_shapefile = csv_geoids - shapefile_geoids

    if in_csv_not_shapefile:
        print(f"   ⚠ Found {len(in_csv_not_shapefile)} GEOID(s) in CSV but not in shapefile:")
        for geoid in sorted(list(in_csv_not_shapefile))[:10]:  # Show first 10
            unmatched_row = puma_df[puma_df['GEOID'] == geoid]
            if len(unmatched_row) > 0:
                state_name = unmatched_row['State_Name'].values[0] if 'State_Name' in unmatched_row.columns and pd.notna(unmatched_row['State_Name'].values[0]) else 'N/A'
                puma_name = unmatched_row['PUMA_Name'].values[0] if 'PUMA_Name' in unmatched_row.columns and pd.notna(unmatched_row['PUMA_Name'].values[0]) else 'No PUMA Name'
                print(f"      - {geoid}: {state_name} ({puma_name})")
        if len(in_csv_not_shapefile) > 10:
            print(f"      ... and {len(in_csv_not_shapefile) - 10} more")

    if in_shapefile_not_csv:
        print(f"   ⚠ Found {len(in_shapefile_not_csv)} GEOID(s) in shapefile but not in CSV (will be excluded)")

    print("\n4. Merging data on GEOID (inner join - only matched records)...")
    # Merge the data on GEOID using inner join to keep only matched records
    integrated_gdf = gdf.merge(
        puma_df,
        on='GEOID',
        how='inner',  # Keep only records that exist in both
        suffixes=('', '_csv')
    )

    print(f"   ✓ Integrated data: {len(integrated_gdf)} records")
    print(f"   ✓ Integrated columns: {len(integrated_gdf.columns)} columns")
    print(f"   ✓ All records have matching spatial and data records")

    # Remove duplicate columns if any (from the merge)
    # Keep the CSV version for data columns, keep original shapefile columns for spatial metadata
    cols_to_keep = []
    seen = set()
    for col in integrated_gdf.columns:
        if col == 'geometry':
            cols_to_keep.append(col)
        elif col.endswith('_csv'):
            # Remove the _csv suffix and keep this version
            base_col = col[:-4]
            if base_col not in seen:
                cols_to_keep.append(col)
                seen.add(base_col)
        elif col not in seen:
            cols_to_keep.append(col)
            seen.add(col)

    # Clean up column names (remove _csv suffix)
    integrated_gdf.columns = [col[:-4] if col.endswith('_csv') else col for col in integrated_gdf.columns]

    # Ensure geometry is the last column (GeoPandas requirement)
    if 'geometry' in integrated_gdf.columns:
        cols = [c for c in integrated_gdf.columns if c != 'geometry'] + ['geometry']
        integrated_gdf = integrated_gdf[cols]

    print("\n5. Saving integrated data as shapefile...")
    # Save to shapefile
    integrated_gdf.to_file(output_path, driver='ESRI Shapefile')

    print(f"   ✓ Integration complete!")
    print(f"   ✓ Output directory: {output_dir}/")
    print(f"   ✓ Output shapefile: {output_shp_name}")
    print(f"   ✓ Total records: {len(integrated_gdf)} (all matched)")

    # Print file size
    shp_files = [f for f in os.listdir(output_dir) if f.startswith(output_shp_name[:-4])]
    total_size = sum(os.path.getsize(os.path.join(output_dir, f)) for f in shp_files) / (1024 * 1024)  # Size in MB
    print(f"   ✓ Total file size: {total_size:.2f} MB")

except ImportError:
    print("   ⚠ geopandas not available. Installing...")
    import subprocess
    import sys
    subprocess.check_call([sys.executable, "-m", "pip", "install", "geopandas"])
    print("   Please re-run this cell after installation.")
except Exception as e:
    print(f"   ✗ Error in PUMA spatial integration: {str(e)}")
    import traceback
    traceback.print_exc()
    print("   Continuing without spatial integration...")




1. Reading shapefile...
   ✓ Shapefile loaded: 2486 records
   ✓ Shapefile columns: ['GISMATCH', 'GISJOIN', 'GEOID', 'State', 'STATEFIP', 'PUMA', 'Name', 'geometry']
   ✓ Coordinate Reference System: PROJCS["USA_Contiguous_Albers_Equal_Area_Conic",GEOGCS["NAD83",DATUM["North_American_Datum_1983",SPHEROID["GRS 1980",6378137,298.257222101,AUTHORITY["EPSG","7019"]],AUTHORITY["EPSG","6269"]],PRIMEM["Greenwich",0,AUTHORITY["EPSG","8901"]],UNIT["degree",0.0174532925199433,AUTHORITY["EPSG","9122"]],AUTHORITY["EPSG","4269"]],PROJECTION["Albers_Conic_Equal_Area"],PARAMETER["latitude_of_center",37.5],PARAMETER["longitude_of_center",-96],PARAMETER["standard_parallel_1",29.5],PARAMETER["standard_parallel_2",45.5],PARAMETER["false_easting",0],PARAMETER["false_northing",0],UNIT["metre",1,AUTHORITY["EPSG","9001"]],AXIS["Easting",EAST],AXIS["Northing",NORTH],AUTHORITY["ESRI","102003"]]
Checking for unmatched records...
   ⚠ Found 1 GEOID(s) in CSV but not in shapefile:
      - 0100400: Alabama ()
   

## Cleaning State

Upload
1. state_2023_5y_all_row E label fix.csv
2. us-state-ansi-fips.csv
3. US_2023_5y_state_RACE_poor.csv
4. the file named 'HexStatesPadded' including 'HexStatesPadded.shp'

Output
1. state_data_2023_cleaned_tableau_with_spatial.csv

In [51]:
import pandas as pd
import numpy as np

state_data = pd.read_csv("state_2023_5y_all_row E label fix.csv")
state_map = pd.read_csv("us-state-ansi-fips.csv", skipinitialspace=True)
state_map.columns = state_map.columns.str.strip()

print(f"Loaded {len(state_data)} state records")
print(f"Loaded {len(state_map)} state mapping records")

Loaded 51 state records
Loaded 51 state mapping records


In [52]:
# Remove duplicate columns and normalize keys

if "STATEFIP.1" in state_data.columns:
    state_data.drop("STATEFIP.1", axis=1, inplace=True)
    print("  Removed duplicate STATEFIP.1 column")

state_data["STATEFIP"] = state_data["STATEFIP"].astype(str).str.zfill(2)

  Removed duplicate STATEFIP.1 column


In [53]:
# Join on state names/abbreviations

state_map["st"] = state_map["st"].astype(str).str.zfill(2)

state_data = state_data.merge(
    state_map[["st", "stname", "stusps"]],
    left_on="STATEFIP",
    right_on="st",
    how="left"
)

state_data.rename(
    columns={
        "STATEFIP": "State_FIPS",
        "stname": "State_Name",
        "stusps": "State_Abbreviation"
    },
    inplace=True
)
state_data.drop(columns=["st"], inplace=True, errors="ignore")

print(f"States found: {state_data['State_Name'].nunique()}")


States found: 51


In [54]:
# Merge race × poor counts + prevalence from external file

race_poor_counts = pd.read_csv("US_2023_5y_state_RACE_poor.csv")
race_poor_counts["STATEFIP"] = race_poor_counts["STATEFIP"].astype(str).str.zfill(2)

# Define all race prefixes
races = ["White", "Black", "AIAN", "Asian", "NHOPI", "Other", "Multi", "Hisp"]

# Build list of columns to keep for each race (keeping original column names):
# 1) {Race}_Poordu_WeightedFrequency
# 2) {Race}_Poordu_MOE_freq
# 3) {Race}_Poordu_percent
# 4) {Race}_Poordu_MOE_percent
# 5) {Race}_Poordu_CV_Percent
race_poor_keep_cols = ["STATEFIP"]

for race in races:
    race_poor_keep_cols.extend([
        f"{race}_Poordu_WeightedFrequency",
        f"{race}_Poordu_MOE_freq",
        f"{race}_Poordu_percent",
        f"{race}_Poordu_MOE_percent",
        f"{race}_Poordu_CV_Percent"
    ])

# Only keep columns that exist in the CSV
race_poor_keep_cols = [col for col in race_poor_keep_cols if col in race_poor_counts.columns]

# Clean numeric values in the race-poor dataset
for col in race_poor_keep_cols:
    if col != "STATEFIP" and col in race_poor_counts.columns:
        race_poor_counts[col] = (
            pd.to_numeric(race_poor_counts[col], errors="coerce")
            .fillna(0)
            .astype(float)
        )

# Merge all columns into the main table (keeping original column names - no renaming)
state_data = state_data.merge(
    race_poor_counts[race_poor_keep_cols],
    left_on="State_FIPS",
    right_on="STATEFIP",
    how="left"
)

state_data.drop(columns=["STATEFIP"], inplace=True)

# Fill missing values with 0 for all race-poor columns
race_poor_data_cols = [col for col in race_poor_keep_cols if col != "STATEFIP"]
state_data[race_poor_data_cols] = state_data[race_poor_data_cols].fillna(0)

# Keep track of count columns for rate calculation (used in next cell)
race_poor_count_cols = [f"{race}_Poordu_WeightedFrequency" for race in races
                        if f"{race}_Poordu_WeightedFrequency" in state_data.columns]

In [55]:
# Build demographic + economic flags

demographic_categories = {
    "White": "White_percent",
    "Black": "Black_percent",
    "AIAN": "AIAN_percent",
    "Asian": "Asian_percent",
    "NHOPI": "NHOPI_percent",
    "Hispanic": "Hispanic_percent",
    "Other": "Other_percent",
    "Multi": "Multi_percent"
}

for category, percent_col in demographic_categories.items():
    if percent_col in state_data.columns:
        state_data[percent_col] = pd.to_numeric(
            state_data[percent_col], errors="coerce"
        ).fillna(0)
        flag_values = (state_data[percent_col] > 0).astype(int)
        state_data[f"{category}_Flag"] = flag_values
        state_data[f"{category}_Category"] = state_data[percent_col].apply(
            lambda x, cat=category: cat if pd.notna(x) and x > 0 else f"Not {cat}"
        )

if "Poordu_percent" in state_data.columns:
    state_data["Poordu_percent"] = pd.to_numeric(
        state_data["Poordu_percent"], errors="coerce"
    ).fillna(0)
    poor_flag = (state_data["Poordu_percent"] > 0).astype(int)
    state_data["Poor_Flag"] = poor_flag
    state_data["Poor_Category"] = state_data["Poordu_percent"].apply(
        lambda x: "Poor" if pd.notna(x) and x > 0 else "Not Poor"
    )

for category in demographic_categories.keys():
    percent_col = demographic_categories[category]
    if percent_col in state_data.columns and "Poordu_percent" in state_data.columns:
        flag_name = f"{category}_Poor_Flag"
        flag_values = (
            (state_data[percent_col] > 0) & (state_data["Poordu_percent"] > 0)
        ).astype(int)
        state_data[flag_name] = flag_values
        print(f"  Created {flag_name}")

print(
    f"\nTotal combined flags created: "
    f"{len([c for c in state_data.columns if '_Flag' in c and '_Poor' in c])}"
)

  Created White_Poor_Flag
  Created Black_Poor_Flag
  Created AIAN_Poor_Flag
  Created Asian_Poor_Flag
  Created NHOPI_Poor_Flag
  Created Hispanic_Poor_Flag
  Created Other_Poor_Flag
  Created Multi_Poor_Flag

Total combined flags created: 8


In [56]:
# Standardize numeric columns

percent_cols = [col for col in state_data.columns if "_percent" in col]
for col in percent_cols:
    state_data[col] = pd.to_numeric(state_data[col], errors="coerce").fillna(0)

freq_cols = [col for col in state_data.columns if "Frequency" in col]
for col in freq_cols:
    state_data[col] = pd.to_numeric(state_data[col], errors="coerce").fillna(0)

for col in percent_cols:
    state_data[col] = state_data[col].round(2)

flag_cols = [
    col for col in state_data.columns
    if "_Flag" in col and "Category" not in col
]
for col in flag_cols:
    state_data[col] = state_data[col].fillna(0).astype(int)
    print(f"  Verified {col}: no nulls, dtype={state_data[col].dtype}")


  Verified White_Flag: no nulls, dtype=int64
  Verified Black_Flag: no nulls, dtype=int64
  Verified AIAN_Flag: no nulls, dtype=int64
  Verified Asian_Flag: no nulls, dtype=int64
  Verified NHOPI_Flag: no nulls, dtype=int64
  Verified Hispanic_Flag: no nulls, dtype=int64
  Verified Other_Flag: no nulls, dtype=int64
  Verified Multi_Flag: no nulls, dtype=int64
  Verified Poor_Flag: no nulls, dtype=int64
  Verified White_Poor_Flag: no nulls, dtype=int64
  Verified Black_Poor_Flag: no nulls, dtype=int64
  Verified AIAN_Poor_Flag: no nulls, dtype=int64
  Verified Asian_Poor_Flag: no nulls, dtype=int64
  Verified NHOPI_Poor_Flag: no nulls, dtype=int64
  Verified Hispanic_Poor_Flag: no nulls, dtype=int64
  Verified Other_Poor_Flag: no nulls, dtype=int64
  Verified Multi_Poor_Flag: no nulls, dtype=int64


In [57]:
# Remove StdErr columns

all_columns = state_data.columns.tolist()
columns_to_drop = []

# Columns to remove
columns_to_drop.extend([
    'StdErrWgtFreq',
    'StdErrofRowPercent',
    'Poordu_StdErrWgtFreq',
    'Poordu_StdErrofRowPercent'
])

# Remove any columns ending with _StdErrWgtFreq or _StdErrofRowPercent
for col in all_columns:
    if col.endswith('_StdErrWgtFreq') or col.endswith('_StdErrofRowPercent'):
        if col not in columns_to_drop:
            columns_to_drop.append(col)

# Remove duplicates and drop columns
columns_to_drop = list(set(columns_to_drop))
columns_to_drop = [col for col in columns_to_drop if col in all_columns]

if len(columns_to_drop) > 0:
    print(f"  Dropping {len(columns_to_drop)} columns: {sorted(columns_to_drop)}")
    state_data = state_data.drop(columns=columns_to_drop, errors='ignore')
    print(f"  Remaining columns: {len(state_data.columns)}")
else:
    print("  No columns to drop")

  Dropping 20 columns: ['AIAN_StdErrWgtFreq', 'AIAN_StdErrofRowPercent', 'Asian_StdErrWgtFreq', 'Asian_StdErrofRowPercent', 'Black_StdErrWgtFreq', 'Black_StdErrofRowPercent', 'Hispanic_StdErrWgtFreq', 'Hispanic_StdErrofRowPercent', 'Multi_StdErrWgtFreq', 'Multi_StdErrofRowPercent', 'NHOPI_StdErrWgtFreq', 'NHOPI_StdErrofRowPercent', 'Other_StdErrWgtFreq', 'Other_StdErrofRowPercent', 'Poordu_StdErrWgtFreq', 'Poordu_StdErrofRowPercent', 'StdErrWgtFreq', 'StdErrofRowPercent', 'White_StdErrWgtFreq', 'White_StdErrofRowPercent']
  Remaining columns: 119


In [58]:
try:
    import geopandas as gpd
    from pathlib import Path

    # Define file paths
    state_shapefile = "HexStatesPadded.shp"
    state_fips_mapping = "us-state-ansi-fips.csv"

    # Read State shapefile
    print("   Reading State hexagon shapefile...")
    state_gdf = gpd.read_file(state_shapefile)
    print(f"   ✓ Loaded {len(state_gdf)} state geometries")
    print(f"   Columns in shapefile: {list(state_gdf.columns)}")

    # Read State data (already loaded as state_data, but ensure it's a DataFrame)
    print("   Using cleaned State data...")
    state_df = state_data.copy()
    print(f"   ✓ Loaded {len(state_df)} state records")
    print(f"   State_FIPS column type: {state_df['State_FIPS'].dtype}")
    print(f"   Sample State_FIPS: {state_df['State_FIPS'].head().tolist()}")

    # Read FIPS mapping for state abbreviations
    print("   Reading FIPS mapping...")
    fips_df = pd.read_csv(state_fips_mapping, skipinitialspace=True)
    fips_df.columns = fips_df.columns.str.strip()
    print(f"   ✓ Loaded FIPS mapping")

    # Check what identifier columns exist in state shapefile
    print("   Checking identifier columns in shapefile...")
    state_id_cols = []
    for col in ['State', 'STATE', 'State_Name', 'STUSPS', 'Abbreviation', 'State_Abbreviation', 'State_Abbr']:
        if col in state_gdf.columns:
            state_id_cols.append(col)
            print(f"   Found column: {col}")

    # Try to match states
    # First, ensure State_FIPS is string with leading zeros
    state_df['State_FIPS'] = state_df['State_FIPS'].astype(str).str.zfill(2)

    # Merge FIPS mapping with state data to get abbreviations (if not already present)
    if 'st' in fips_df.columns:
        fips_df['st'] = fips_df['st'].astype(str).str.zfill(2)
        state_df = state_df.merge(
            fips_df[['st', 'stusps']],
            left_on='State_FIPS',
            right_on='st',
            how='left'
        )
        # Fill State_Abbreviation if missing
        if 'State_Abbreviation' in state_df.columns:
            state_df['State_Abbreviation'] = state_df['State_Abbreviation'].fillna(state_df['stusps'])
        else:
            state_df['State_Abbreviation'] = state_df['stusps']
        state_df.drop(columns=['st'], inplace=True, errors='ignore')

    # Strategy 1: Merge by State_Abbreviation / State_Abbr
    if 'State_Abbr' in state_gdf.columns or 'Abbreviation' in state_gdf.columns or 'STUSPS' in state_gdf.columns:
        abbrev_col = None
        if 'State_Abbr' in state_gdf.columns:
            abbrev_col = 'State_Abbr'
        elif 'Abbreviation' in state_gdf.columns:
            abbrev_col = 'Abbreviation'
        elif 'STUSPS' in state_gdf.columns:
            abbrev_col = 'STUSPS'

        if abbrev_col:
            print(f"   Merging using {abbrev_col}...")
            state_gdf_merge = state_gdf.copy()
            if abbrev_col != 'State_Abbreviation':
                state_gdf_merge['State_Abbreviation'] = state_gdf_merge[abbrev_col]

            merged = state_gdf_merge.merge(
                state_df,
                on='State_Abbreviation',
                how='outer',
                suffixes=('_shape', '_data')
            )
            print(f"   ✓ Merged using State_Abbreviation: {len(merged)} records")

            state_integrated = merged
        else:
            raise ValueError("Could not determine abbreviation column for merging")
    else:
        print("   ⚠ Could not find State abbreviation column. Available columns:")
        print(f"   Shapefile: {list(state_gdf.columns)}")
        print(f"   Data: {list(state_df.columns)}")
        raise ValueError("Could not find State abbreviation column for merging")

    # Process the merged data
    print(f"   ✓ Merged dataset contains {len(state_integrated)} records")
    print(f"   Records with geometry: {state_integrated['geometry'].notna().sum()}")

    # Check for data columns
    data_cols = [col for col in state_integrated.columns if 'DU' in col or 'Poor' in col or 'Frequency' in col]
    if data_cols:
        print(f"   Records with data: {state_integrated[data_cols[0]].notna().sum()}")

    # Convert geometry to WKT (Well-Known Text) format for CSV export
    print("   Converting geometry to WKT format...")
    state_integrated['Geometry_WKT'] = state_integrated['geometry'].apply(lambda x: x.wkt if x is not None else None)

    # Drop the geometry column (we have WKT now)
    state_integrated_csv = state_integrated.drop(columns=['geometry'])

    # Sort by State_FIPS (convert to numeric for proper sorting)
    print("   Sorting by State_FIPS...")
    state_integrated_csv['State_FIPS_num'] = pd.to_numeric(state_integrated_csv['State_FIPS'], errors='coerce')
    state_integrated_csv = state_integrated_csv.sort_values('State_FIPS_num').drop(columns=['State_FIPS_num'])

    # Save integrated State data as CSV with WKT geometry
    state_output_csv = "state_data_2023_cleaned_tableau_with_spatial.csv"

    print(f"   Saving to CSV with WKT geometry: {state_output_csv}")
    state_integrated_csv.to_csv(state_output_csv, index=False)

    print("   ✓ State integration complete!")
    print(f"   Final output: {len(state_integrated_csv)} records with {len(state_integrated_csv.columns)} columns")
    print(f"   Geometry column: Geometry_WKT (WKT format)")

except ImportError:
    print("   ⚠ geopandas not available. Installing...")
    import subprocess
    import sys
    subprocess.check_call([sys.executable, "-m", "pip", "install", "geopandas"])
    print("   Please re-run this cell after installation.")
except Exception as e:
    print(f"   ✗ Error in State integration: {str(e)}")
    import traceback
    traceback.print_exc()
    print("   Continuing with non-spatial export only...")



   Reading State hexagon shapefile...
   ✓ Loaded 51 state geometries
   Columns in shapefile: ['State', 'State_Abbr', 'geometry']
   Using cleaned State data...
   ✓ Loaded 51 state records
   State_FIPS column type: object
   Sample State_FIPS: ['01', '02', '04', '05', '06']
   Reading FIPS mapping...
   ✓ Loaded FIPS mapping
   Checking identifier columns in shapefile...
   Found column: State
   Found column: State_Abbr
   Merging using State_Abbr...
   ✓ Merged using State_Abbreviation: 51 records
   ✓ Merged dataset contains 51 records
   Records with geometry: 51
   Records with data: 51
   Converting geometry to WKT format...
   Sorting by State_FIPS...
   Saving to CSV with WKT geometry: state_data_2023_cleaned_tableau_with_spatial.csv
   ✓ State integration complete!
   Final output: 51 records with 123 columns
   Geometry column: Geometry_WKT (WKT format)


In [60]:
# Quality checks

print(f"\nTotal records: {len(state_integrated_csv)}")
print(f"Total columns: {len(state_integrated_csv.columns)}")

print("\nGeographic coverage:")
print(f"  States: {state_integrated_csv['State_Name'].nunique()}")

print("\nMissing values:")
missing = state_integrated_csv.isnull().sum()
missing = missing[missing > 0]
if len(missing) > 0:
    print(missing.head(10))
else:
    print("  No missing values in key columns")

print("\nDemographic categories with data:")
for category in demographic_categories.keys():
    flag_col = f"{category}_Flag"
    if flag_col in state_integrated_csv.columns:
        count = state_integrated_csv[flag_col].sum()
        print(f"  {category}: {count} states ({count/len(state_integrated_csv)*100:.1f}%)")

print("\nCombined flags summary:")
poor_flags = [c for c in state_integrated_csv.columns if "_Poor_Flag" in c]
print(f"  Demographic × Poor flags: {len(poor_flags)}")
print(f"  Total intersection flags: {len(poor_flags)}")

print("\nSample of cleaned data:")
print(
    state_integrated_csv[
        [
            "State_FIPS",
            "State_Name",
            "State_Abbreviation",
            "DUpercent_prevalenceinSTATE",
            "Poordu_percent",
            "White_percent",
            "Hispanic_percent",
            "White_Poordu_WeightedFrequency",
            "White_Poordu_MOE_freq",
            "White_Poordu_percent"
        ]
    ].head(10)
)

print(state_integrated_csv.columns)




Total records: 51
Total columns: 123

Geographic coverage:
  States: 51

Missing values:
  No missing values in key columns

Demographic categories with data:
  White: 51 states (100.0%)
  Black: 49 states (96.1%)
  AIAN: 48 states (94.1%)
  Asian: 50 states (98.0%)
  NHOPI: 39 states (76.5%)
  Hispanic: 51 states (100.0%)
  Other: 51 states (100.0%)
  Multi: 51 states (100.0%)

Combined flags summary:
  Demographic × Poor flags: 8
  Total intersection flags: 8

Sample of cleaned data:
  State_FIPS            State_Name State_Abbreviation  \
1         01               Alabama                 AL   
0         02                Alaska                 AK   
3         04               Arizona                 AZ   
2         05              Arkansas                 AR   
4         06            California                 CA   
5         08              Colorado                 CO   
6         09           Connecticut                 CT   
8         10              Delaware                 D